**Run evidence** · task `02d_supplier_risk` · run `936837607229803` · result **SUCCESS** · executed 2026-10-19 17:07 UTC on serverless.

Exported from the Databricks job run with `_helper/export_run_evidence.py`. Source notebook: `02_build_gold/02d_supplier_risk`.

# 02d — Workflow D → SUPPLIER_RISK with a Lakeflow Declarative Pipeline

Enrichment layer joined at query time to flag delivery risk. Time-sliced so a PO
placed on a given date can be evaluated against the risk window that was active
*at that date* (drives Q8).

Unlike the other gold tables, this slice is **ingested from files**: SAP Ariba
Supplier Risk exports land as JSON in the UC volume `ariba_landing` (written by
`_helper/01_simulate_raw_tables`). The pipeline `02d_supplier_risk_pipeline.sql`:

```
/Volumes/<catalog>/<schema>/ariba_landing/supplier_risk/*.json
    │  Auto Loader (read_files, incremental)
    ▼
RAW_SUPPLIER_RISK   streaming table · Ariba fields → SAP-style columns · lineage columns
    │  latest snapshot · RI join to SUPPLIER · expectations · PK (LIFNR, VALID_FROM)
    ▼
SUPPLIER_RISK       materialized view (gold)
```

**Steps (spec §4, Workflow D):**
1. Take the raw Ariba risk records (Auto Loader → `RAW_SUPPLIER_RISK`).
2. Validate LIFNR values exist in Workflow B's SUPPLIER output (referential
   integrity) — we keep only vendors that exist in SUPPLIER.
3. No merging — this table stands alone as an enrichment layer.

This notebook creates (or updates) the pipeline, runs one update and prints the
evidence: update state, row counts, expectation results and the primary key. In the
Lakeflow Job the same pipeline runs as the `gold_supplier_risk` pipeline task.

In [ ]:
%run ../00_config

[00_config] Using namespace: max_kohler_demo_catalog.purchasing_demo


DemoConfig(
  catalog       = 'max_kohler_demo_catalog'
  schema        = 'purchasing_demo'
  namespace     = 'max_kohler_demo_catalog.purchasing_demo'
  warehouse_id  = 'd7a3d1cfbba73c0a'
  model         = 'databricks-claude-sonnet-4-5'
  genie_space_1 = 'SAP Purchasing — Sourcing & Supplier Risk'
  genie_space_2 = 'SAP Purchasing — Purchase Order Analytics'
  mas_endpoint  = 'sap-purchasing-mas'
  app_name      = 'sap-purchasing-app'
  job_name      = 'sap_purchasing_lakeflow_pipeline'
  lakebase      = 'sap-purchasing-ops'/'purchasing_ops' (UC catalog 'sap_purchasing_ops')
)


In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
pipeline_id = ensure_supplier_risk_pipeline()
print(f"Pipeline UI: {w.config.host}/pipelines/{pipeline_id}")

✓ Updated pipeline 'sap_purchasing_ariba_supplier_risk_purchasing_demo' (1146b932-5e41-4e0e-8c98-5a73903c2dac)
Pipeline UI: https://fe-sandbox-serverless-sandbox-gs5ou3.cloud.databricks.com/pipelines/1146b932-5e41-4e0e-8c98-5a73903c2dac


## Run one pipeline update and wait

In [ ]:
update_id = w.pipelines.start_update(pipeline_id=pipeline_id).update_id
print(f"Started update {update_id}")
while True:
    state = w.pipelines.get_update(pipeline_id=pipeline_id, update_id=update_id).update.state.value
    if state in ("COMPLETED", "FAILED", "CANCELED"):
        break
    time.sleep(15)
print(f"Pipeline update {update_id}: {state}")
if state != "COMPLETED":
    errors = [e.message for e in w.pipelines.list_pipeline_events(pipeline_id=pipeline_id, max_results=50)
              if e.level and e.level.value == "ERROR"]
    raise RuntimeError(f"Pipeline update {state}: {errors[:3]}")

Started update 59ae4344-af5c-484b-872b-e69f1ed930a5
Pipeline update 59ae4344-af5c-484b-872b-e69f1ed930a5: COMPLETED


## Evidence: what the pipeline produced

In [ ]:
NS = CFG.namespace
files = dbutils.fs.ls(CFG.ariba_landing_path)
raw = spark.table(CFG.fq("RAW_SUPPLIER_RISK"))
latest = raw.agg({"EXPORT_DATE": "max"}).first()[0]
raw_latest = raw.filter(f"EXPORT_DATE = '{latest}'").count()
cnt = spark.table(CFG.fq("SUPPLIER_RISK")).count()
active = (spark.table(CFG.fq("SUPPLIER_RISK"))
          .filter("DELIVERY_RISK_FLAG = 'X' AND VALID_FROM <= current_date() "
                  "AND VALID_TO >= current_date()")
          .select("LIFNR").distinct().count())

print(f"Ariba export files in volume:   {len(files)}  {[f.name for f in files]}")
print(f"RAW_SUPPLIER_RISK rows:         {raw.count():,}  (latest export {latest}: {raw_latest})")
print(f"SUPPLIER_RISK rows (gold):      {cnt:,}  | not in gold: {raw_latest - cnt}")
print(f"Vendors with ACTIVE delivery-risk flag today: {active}")

display(spark.sql(f"""SELECT table_name, table_type, comment
                      FROM {CFG.catalog}.information_schema.tables
                      WHERE table_schema = '{CFG.schema}'
                        AND lower(table_name) IN ('raw_supplier_risk', 'supplier_risk')"""))

Ariba export files in volume:   1  ['supplier_risk_export_20261019.json']
RAW_SUPPLIER_RISK rows:         34  (latest export 2026-10-19: 34)
SUPPLIER_RISK rows (gold):      32  | not in gold: 2
Vendors with ACTIVE delivery-risk flag today: 7


table_name,table_type,comment
supplier_risk,MATERIALIZED_VIEW,"Gold: time-sliced supplier risk classification from SAP Ariba Supplier Risk. One row per vendor (LIFNR) and validity window (VALID_FROM..VALID_TO). DELIVERY_RISK_FLAG='X' marks delivery risk. Join at query time; for 'active' risk, filter VALID_FROM<=date<=VALID_TO. Built by the Lakeflow pipeline from the latest Ariba export, keeping only vendors that exist in SUPPLIER."
raw_supplier_risk,STREAMING_TABLE,"Raw: SAP Ariba Supplier Risk export records, ingested incrementally by Auto Loader from JSON files in the ariba_landing volume. One row per vendor, risk window and export day. Ariba fields mapped to SAP-style columns; _SOURCE_FILE and _INGESTED_AT give lineage."


### Expectations (data quality) from the pipeline event log

In [ ]:
display(spark.sql(f"""
  SELECT e.dataset, e.name AS expectation, e.passed_records, e.failed_records
  FROM (
    SELECT explode(from_json(details:flow_progress.data_quality.expectations,
                   'array<struct<name:string,dataset:string,passed_records:bigint,failed_records:bigint>>')) AS e
    FROM {CFG.fq(CFG.ariba_event_log)}
    WHERE event_type = 'flow_progress' AND origin.update_id = '{update_id}'
      AND details:flow_progress.data_quality IS NOT NULL)
  ORDER BY e.dataset, e.name"""))

dataset,expectation,passed_records,failed_records
max_kohler_demo_catalog.purchasing_demo.supplier_risk,valid_score,32,1
max_kohler_demo_catalog.purchasing_demo.supplier_risk,valid_window,33,0


### Primary key declared on the materialized view

In [ ]:
display(spark.sql(f"""
  SELECT tc.table_name, tc.constraint_name, tc.constraint_type,
         concat_ws(', ', collect_list(kcu.column_name)) AS columns
  FROM {CFG.catalog}.information_schema.table_constraints tc
  JOIN {CFG.catalog}.information_schema.key_column_usage kcu
    ON tc.constraint_name = kcu.constraint_name AND tc.table_schema = kcu.table_schema
  WHERE tc.table_schema = '{CFG.schema}' AND lower(tc.table_name) = 'supplier_risk'
  GROUP BY ALL"""))

display(spark.table(CFG.fq("SUPPLIER_RISK")).orderBy("LIFNR", "VALID_FROM").limit(10))

table_name,constraint_name,constraint_type,columns
supplier_risk,pk_risk,PRIMARY KEY,"VALID_FROM, LIFNR"


LIFNR,RISK_CATEGORY,DELIVERY_RISK_FLAG,RISK_SCORE,VALID_FROM,VALID_TO
V200000,QUALITY,,35,2026-01-24,2027-05-01
V200001,QUALITY,,43,2026-04-16,2027-06-26
V200002,COMPLIANCE,,25,2025-11-16,2027-07-12
V200003,QUALITY,,13,2025-11-09,2027-07-25
V200004,COMPLIANCE,,40,2026-04-25,2027-06-29
V200005,FINANCIAL,,34,2026-04-21,2027-06-02
V200006,DELIVERY,X,84,2025-01-31,2026-02-25
V200006,DELIVERY,X,78,2026-05-10,2027-06-14
V200007,DELIVERY,X,82,2024-11-04,2026-01-29
V200007,DELIVERY,X,80,2026-02-28,2027-11-08


In [ ]:
dbutils.notebook.exit(f"02d_supplier_risk: SUCCESS — pipeline {pipeline_id} update {update_id} "
                      f"COMPLETED, {cnt} gold rows.")

Notebook exited: 02d_supplier_risk: SUCCESS — pipeline 1146b932-5e41-4e0e-8c98-5a73903c2dac update 59ae4344-af5c-484b-872b-e69f1ed930a5 COMPLETED, 32 gold rows.
